In [1]:
!pip uninstall -y transformers
!pip uninstall -y sentence-transformers

Found existing installation: transformers 4.52.2
Uninstalling transformers-4.52.2:
  Successfully uninstalled transformers-4.52.2
Found existing installation: sentence-transformers 4.1.0
Uninstalling sentence-transformers-4.1.0:
  Successfully uninstalled sentence-transformers-4.1.0


In [2]:
!pip install transformers==4.40.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1


In [3]:
!pip install transformers datasets scikit-learn --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.1
    Uninstalling transformers-4.40.1:
      Successfully uninstalled transformers-4.40.1
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR:

In [4]:
# Install required libraries
!pip install --quiet transformers datasets faiss-cpu accelerate

import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    RagTokenizer,
    RagRetriever,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.2 MB/s eta 0:00:00


In [5]:
# Imports for DPR + FAISS
import torch
import faiss
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer
)

## RAG Retrieval:

In [6]:
# Load & prepare the knowledge corpus:

import pandas as pd

# Load DDI interactions and build a text field
df1 = pd.read_csv("/content/DDI_data (3).csv")
df1 = df1[["drug1_name", "drug2_name", "interaction_type"]].dropna()
df1["source"] = "DDI"
df1["text"] = (
    df1["drug1_name"]
    + " may interact with "
    + df1["drug2_name"]
    + ": "
    + df1["interaction_type"]
)
df1.rename(
    columns={
        "drug1_name": "drug1",
        "drug2_name": "drug2",
        "interaction_type": "interaction",
    },
    inplace=True,
)

# Load EML medical uses and build a text field
df3 = pd.read_excel("/content/EML export (2).xlsx")
df3.columns = df3.columns.str.strip()
df3 = df3[["Medicine name", "Indication"]].dropna()
df3["source"] = "EML"
df3["text"] = df3["Medicine name"] + " is used for: " + df3["Indication"]
df3.rename(
    columns={"Medicine name": "medicine_name", "Indication": "indication"},
    inplace=True,
)

# Combine both sources into one DataFrame and drop duplicate sentences
knowledge_df = pd.concat([df1, df3], ignore_index=True)
knowledge_df.drop_duplicates(subset=["text"], inplace=True)

# Find drug names that appear in both sources
ddi_names = df1["drug1"].str.lower().unique()
eml_names = df3["medicine_name"].str.lower().unique()
common_names = set(ddi_names).intersection(eml_names)

# Filter only the overlapping entries to inspect them
overlapping_texts = knowledge_df[
    (
        (knowledge_df["source"] == "DDI")
        & knowledge_df["drug1"].str.lower().isin(common_names)
    )
    | (
        (knowledge_df["source"] == "EML")
        & knowledge_df["medicine_name"].str.lower().isin(common_names)
    )
]

print("Number of overlapping entries in corpus:", len(overlapping_texts))
display(overlapping_texts)

# Export the full corpus
knowledge_df.to_excel("knowledge_corpus.xlsx", index=False)

# Export only the overlapping entries
overlapping_texts.to_excel("overlapping_entries.xlsx", index=False)

Number of overlapping entries in corpus: 71983


,drug1,drug2,interaction,source,text,medicine_name,indication
225,Goserelin,Dofetilide,QTc-prolonging activities,DDI,Goserelin may interact with Dofetilide: QTc-pr...,NaN,NaN
226,Goserelin,Citalopram,QTc-prolonging activities,DDI,Goserelin may interact with Citalopram: QTc-pr...,NaN,NaN
227,Goserelin,Ziprasidone,QTc-prolonging activities,DDI,Goserelin may interact with Ziprasidone: QTc-p...,NaN,NaN
228,Goserelin,Anagrelide,QTc-prolonging activities,DDI,Goserelin may interact with Anagrelide: QTc-pr...,NaN,NaN
229,Goserelin,Disopyramide,QTc-prolonging activities,DDI,Goserelin may interact with Disopyramide: QTc-...,NaN,NaN
...,...,...,...,...,...,...,...
224261,NaN,NaN,NaN,EML,warfarin is used for: Atrial fibrillation,warfarin,Atrial fibrillation
224264,NaN,NaN,NaN,EML,xylometazoline is used for: Nasal congestion,xylometazoline,Nasal congestion
224268,NaN,NaN,NaN,EML,zidovudine is used for: Human immunodeficiency...,zidovudine,Human immunodeficiency virus disease without m...
224270,NaN,NaN,NaN,EML,zoledronic acid is used for: Malignant neoplas...,zoledronic acid,Malignant neoplasm metastasis in bone or bone ...


## Prepare and index RAG knowledge corpus:

In [7]:
import torch
from datasets import Dataset
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    RagTokenizer,
    RagRetriever
)
import pandas as pd
import faiss
import numpy as np

# Load corpus and build HF Dataset with title/text
knowledge_df = pd.read_excel("/content/knowledge_corpus.xlsx")
passages = knowledge_df["text"].tolist()
passages_ds = Dataset.from_dict({
    "title": [f"doc_{i}" for i in range(len(passages))],
    "text":  passages
})

# Compute DPR embeddings for each passage
device = "cuda" if torch.cuda.is_available() else "cpu"
ctx_tok = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_enc = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base").to(device)

def embed_batch(batch):
    toks = ctx_tok(batch["text"], truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = ctx_enc(**toks)
    return {"embeddings": out.pooler_output.cpu().numpy()}

passages_ds = passages_ds.map(
    embed_batch,
    batched=True,
    batch_size=32,
    load_from_cache_file=False
)

#Save HF Dataset to disk
passages_ds.save_to_disk("hf_knowledge_dataset")

# Build & write the Faiss index
embs = np.vstack(passages_ds["embeddings"]).astype("float32")  # ← ודאי float32
faiss.normalize_L2(embs)
d = embs.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embs)
faiss.write_index(index, "/content/hf_knowledge_index.faiss")

#instantiate RagRetriever
rag_ckpt = "facebook/rag-sequence-base"
tokenizer_rag = RagTokenizer.from_pretrained(rag_ckpt)
retriever = RagRetriever.from_pretrained(
    rag_ckpt,
    index_name="custom",
    passages_path="hf_knowledge_dataset",
    index_path="/content/hf_knowledge_index.faiss"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may res

In [8]:
import pandas as pd
import faiss

knowledge_df = pd.read_excel("/content/knowledge_corpus.xlsx")
passages = knowledge_df["text"].tolist()

# load the index
index = faiss.read_index("/content/hf_knowledge_index.faiss")

In [9]:
def retrieve_topk(question: str, k: int = 3):
    toks = q_tok(question, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = q_enc(**toks)
    q_vec = out.pooler_output.cpu().numpy()
    faiss.normalize_L2(q_vec)

    D, I = index.search(q_vec, k)
    return [passages[i] for i in I[0].tolist()]

## Load and split classification data:

In [10]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load labeled questions
df = pd.read_excel("/content/MedInfo2019-QA-MedicationsFINAL (1) (4).xlsx")

# print column names to confirm
print("Columns in df:", df.columns.tolist())

df = df.rename(columns={
    "Question": "question",
    "Risk_Level": "label_text"
})

# Map textual labels to integer IDs
label_map = {"General": 0, "Critical": 1}
df["label"] = df["label_text"].map(label_map)

# Sanity check
print(df[["question","label_text","label"]].head())

# Stratified 80/20 split
train_df, eval_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

# Convert to Hugging Face Dataset objects
# We only need the two columns: 'question' and 'label'
train_ds = Dataset.from_pandas(
    train_df[["question","label"]].reset_index(drop=True)
)
eval_ds = Dataset.from_pandas(
    eval_df[["question","label"]].reset_index(drop=True)
)

print(f"Train size: {len(train_ds)}  Eval size: {len(eval_ds)}")

Columns in df: ['Question', 'Risk_Level', 'Focus (Drug)', 'Question Type', 'Answer', 'Section Title', 'URL']
                                            question label_text  label
0  how does rivatigmine and otc sleep medicine in...   Critical      1
1                   how does valium affect the brain    General      0
2                                   what is morphine    General      0
3            what are the milligrams for oxycodone e    General      0
4     81% aspirin contain resin and shellac in it. ?    General      0
Train size: 524  Eval size: 131


## Define retrieval + augmentation preprocessing:

In [11]:
from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer

# Load a DPR Question Encoder & its tokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
q_tok = DPRQuestionEncoderTokenizer.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)
q_enc = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to(device)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [12]:
import faiss

# Load the FAISS index
index = faiss.read_index("/content/hf_knowledge_index.faiss")

combined_train = []
train_labels   = []

for ex in train_ds:
    q   = ex["question"]
    lab = ex["label"]

    # DPR encode
    q_inputs = q_tok(q, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        q_out = q_enc(**q_inputs)
    q_vec = q_out.pooler_output.cpu().numpy()

    # Faiss search
    D, I = index.search(q_vec, 3)
    idxs = I[0].tolist()

    ctxs = [passages[i] for i in idxs]

    # combine
    combined_train.append(q + " " + " ".join(ctxs))
    train_labels.append(lab)

# same for eval
combined_eval = []
eval_labels   = []

for ex in eval_ds:
    q   = ex["question"]
    lab = ex["label"]

    q_inputs = q_tok(q, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        q_out = q_enc(**q_inputs)
    q_vec = q_out.pooler_output.cpu().numpy()

    D, I = index.search(q_vec, 3)
    idxs = I[0].tolist()

    ctxs = [passages[i] for i in idxs]
    combined_eval.append(q + " " + " ".join(ctxs))
    eval_labels.append(lab)

# build fresh pandas + HF Datasets
import pandas as pd
from datasets import Dataset

df_train = pd.DataFrame({
    "combined_text": combined_train,
    "label":         train_labels
})
df_eval = pd.DataFrame({
    "combined_text": combined_eval,
    "label":         eval_labels
})

from datasets import Dataset

train_ds = Dataset.from_pandas(df_train)
eval_ds  = Dataset.from_pandas(df_eval)

print(train_ds.column_names)
print(eval_ds.column_names)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

['combined_text', 'label']
['combined_text', 'label']


## Tokenize the augmented inputs:

In [13]:
# Preservation of the original texts before tokenization
df_train = train_ds.to_pandas()[["combined_text", "label"]]
df_eval = eval_ds.to_pandas()[["combined_text", "label"]]

df_train.to_excel("train_rag_prepared.xlsx", index=False)
df_eval.to_excel("eval_rag_prepared.xlsx", index=False)

In [14]:
# Fine-Tuning (BlueBERT + class weights) on the pre-prepared RAG data

# Disable wandb (if installed) before importing transformers
import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import torch
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score
from sklearn.utils import resample
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Load the RAG files
df_train0 = pd.read_excel("train_rag_prepared.xlsx")
df_eval0  = pd.read_excel("eval_rag_prepared.xlsx")

# Perform oversampling
df_general  = df_train0[df_train0["label"] == 0]
df_critical = df_train0[df_train0["label"] == 1]

# Oversample the smaller “Critical” class to match the size of “General”
df_critical_upsampled = resample(
    df_critical,
    replace=True,
    n_samples=len(df_general),
    random_state=42
)

# Then concatenate with all of df_general (unchanged)
df_train = pd.concat([df_general, df_critical_upsampled]) \
             .sample(frac=1, random_state=42) \
             .reset_index(drop=True)

print("Train distribution after oversampling:")
print(df_train["label"].value_counts(), "\n")

# Convert to HuggingFace Dataset format
train_ds = Dataset.from_pandas(df_train)
eval_ds  = Dataset.from_pandas(df_eval0)

# Remove any auto-generated index columns if present
for ds in (train_ds, eval_ds):
    if "__index_level_0__" in ds.column_names:
        ds = ds.remove_columns("__index_level_0__")

dataset_dict = DatasetDict({"train": train_ds, "eval": eval_ds})

# Tokenization (note: combined_text already contains "[CONTEXT]: … [QUESTION]: …")
model_name = "bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12"
tokenizer  = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["combined_text"],  # The enriched RAG text
        padding="max_length",
        truncation=True,
        max_length=384
    )

tokenized_datasets = dataset_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=["combined_text"]
)

# Compute class weights
train_labels = df_train["label"].tolist()
n_total   = len(train_labels)
n_general  = sum(1 for lbl in train_labels if lbl == 0)
n_critical = sum(1 for lbl in train_labels if lbl == 1)

weight_for_general  = n_total / (2 * n_general) if n_general>0 else 1.0
weight_for_critical = n_total / (2 * n_critical) if n_critical>0 else 1.0
print(f"Class weights → General: {weight_for_general:.4f}, Critical: {weight_for_critical:.4f}")

class_weights = torch.tensor([weight_for_general, weight_for_critical])

# Load the base BlueBERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Define custom loss function with class weights
def weighted_loss_fn(logits, labels):
    loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
    return loss_fct(logits, labels)

# Define evaluation metrics (Precision/Recall/F1/Accuracy)
def compute_metrics(p):
    preds  = p.predictions.argmax(-1)
    labels = p.label_ids
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0,1]
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": float(acc),
        "precision_general": float(precision[0]),
        "recall_general":    float(recall[0]),
        "f1_general":        float(f1[0]),
        "precision_critical":float(precision[1]),
        "recall_critical":   float(recall[1]),
        "f1_critical":       float(f1[1])
    }

# Subclass Trainer to override compute_loss (using class weights)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs.get("input_ids"),
            attention_mask=inputs.get("attention_mask"),
            labels=None
        )
        logits = outputs.logits
        loss = weighted_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results_bluebert",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_dir="./logs_bluebert",
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)

# Instantiate the Trainer using the subclass
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["eval"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train the model
train_result = trainer.train()
trainer.save_model("./bluebert_clinical_classifier")


# Evaluate performance on the Eval set
predictions = trainer.predict(tokenized_datasets["eval"])
preds = predictions.predictions.argmax(axis=1)
labels = predictions.label_ids

print("\nClassification Report (Eval):")
print(classification_report(labels, preds, target_names=["General","Critical"]))

Train distribution after oversampling:
label
0    434
1    434
Name: count, dtype: int64 



config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Map:   0%|          | 0/868 [00:00<?, ? examples/s]

Map:   0%|          | 0/131 [00:00<?, ? examples/s]

...

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-14-28a5859b54e4>:142: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


Step,Training Loss
50,0.685300
100,0.593800
150,0.505800
200,0.382700
250,0.277900
300,0.205900
350,0.161900
400,0.129200
450,0.131900
500,0.122600


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]


Classification Report (Eval):
              precision    recall  f1-score   support

     General       0.87      0.86      0.87       109
    Critical       0.35      0.36      0.36        22

    accuracy                           0.78       131
   macro avg       0.61      0.61      0.61       131
weighted avg       0.78      0.78      0.78       131



In [15]:
# Turn off wandb (if installed) so that the API Key request does not pop up.
import os
os.environ["WANDB_DISABLED"] = "true"

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    MarianMTModel,
    MarianTokenizer
)

def focal_loss(logits: torch.Tensor, labels: torch.Tensor, alpha: float = 0.25, gamma: float = 2.0) -> torch.Tensor:
    ce = F.cross_entropy(logits, labels, reduction="none")
    pt = torch.exp(-ce)
    focal = alpha * (1 - pt) ** gamma * ce
    return focal.mean()

df_orig = pd.read_excel("train_rag_prepared.xlsx")

model_name_en_fr = "Helsinki-NLP/opus-mt-en-fr"
model_name_fr_en = "Helsinki-NLP/opus-mt-fr-en"

tokenizer_en_fr = MarianTokenizer.from_pretrained(model_name_en_fr)
model_en_fr     = MarianMTModel.from_pretrained(model_name_en_fr).to("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_fr_en = MarianTokenizer.from_pretrained(model_name_fr_en)
model_fr_en     = MarianMTModel.from_pretrained(model_name_fr_en).to("cuda" if torch.cuda.is_available() else "cpu")

def back_translate(text: str,
                   tokenizer_src: MarianTokenizer, model_src: MarianMTModel,
                   tokenizer_tgt: MarianTokenizer, model_tgt: MarianMTModel) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch1 = tokenizer_src.prepare_seq2seq_batch([text], return_tensors="pt").to(device)
    trans1 = model_src.generate(**batch1)
    fr_text = tokenizer_src.batch_decode(trans1, skip_special_tokens=True)[0]
    batch2 = tokenizer_tgt.prepare_seq2seq_batch([fr_text], return_tensors="pt").to(device)
    trans2 = model_tgt.generate(**batch2)
    en_back = tokenizer_tgt.batch_decode(trans2, skip_special_tokens=True)[0]
    return en_back

df_critical = df_orig[df_orig["label"] == 1].reset_index(drop=True)

augmented_rows = []
print("Start Augmentation for", len(df_critical), "Critical Examples …")
for idx, row in tqdm(df_critical.iterrows(), total=len(df_critical)):
    original = row["combined_text"]
    try:
        bt = back_translate(original, tokenizer_en_fr, model_en_fr, tokenizer_fr_en, model_fr_en)
    except Exception:
        bt = original
    augmented_rows.append({
        "combined_text": bt,
        "label": row["label"]
    })

df_augmented = pd.concat([
    df_orig,
    pd.DataFrame(augmented_rows)
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("\n--- Distribution before Augmentation ---")
print(df_orig["label"].value_counts().rename({0:"General",1:"Critical"}))
print("\n--- Distribution after Augmentation ---")
print(df_augmented["label"].value_counts().rename({0:"General",1:"Critical"}))

# --- Stratified K-Fold Cross-Validation setup ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_reports = []
f1_critical_scores = []

hf_tokenizer = AutoTokenizer.from_pretrained("bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12")

for fold_idx, (train_idx, eval_idx) in enumerate(skf.split(df_augmented, df_augmented["label"]), start=1):
    print(f"\n\n========== Fold {fold_idx} / 5 ==========\n")
    df_train_fold = df_augmented.iloc[train_idx].reset_index(drop=True)
    df_eval_fold  = df_augmented.iloc[eval_idx].reset_index(drop=True)

    train_ds = Dataset.from_pandas(df_train_fold[["combined_text", "label"]])
    eval_ds  = Dataset.from_pandas(df_eval_fold[["combined_text", "label"]])
    for ds in (train_ds, eval_ds):
        if "__index_level_0__" in ds.column_names:
            ds.remove_columns("__index_level_0__")

    def tokenize_fn(examples):
        return hf_tokenizer(
            examples["combined_text"],
            padding="max_length",
            truncation=True,
            max_length=384
        )

    tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["combined_text"])
    tokenized_eval  = eval_ds.map(tokenize_fn, batched=True, remove_columns=["combined_text"])

    tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    tokenized_eval.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    labels_train_fold = df_train_fold["label"].tolist()
    n_tot = len(labels_train_fold)
    n_gen = sum(1 for l in labels_train_fold if l == 0)
    n_cr = sum(1 for l in labels_train_fold if l == 1)
    weight_gen = n_tot / (2 * n_gen) if n_gen > 0 else 1.0
    weight_cr = n_tot / (2 * n_cr) if n_cr > 0 else 1.0
    class_weights = torch.tensor([weight_gen, weight_cr])

    model = AutoModelForSequenceClassification.from_pretrained(
        "bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12",
        num_labels=2
    )

    def compute_metrics(p):
        preds  = p.predictions.argmax(-1)
        labels = p.label_ids
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average=None, labels=[0,1]
        )
        acc = accuracy_score(labels, preds)
        return {
            "accuracy": float(acc),
            "precision_general": float(precision[0]),
            "recall_general":    float(recall[0]),
            "f1_general":        float(f1[0]),
            "precision_critical":float(precision[1]),
            "recall_critical":   float(recall[1]),
            "f1_critical":       float(f1[1])
        }

    class FocalTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(
                input_ids=inputs.get("input_ids"),
                attention_mask=inputs.get("attention_mask"),
                labels=None
            )
            logits = outputs.logits
            loss = focal_loss(logits, labels, alpha=0.25, gamma=2.0)
            return (loss, outputs) if return_outputs else loss

    training_args = TrainingArguments(
        output_dir=f"./results_fold{fold_idx}",
        num_train_epochs=8,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_dir=f"./logs_fold{fold_idx}",
        logging_steps=50,
        save_steps=200,
        save_total_limit=1,
        report_to="none"
    )

    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        tokenizer=hf_tokenizer,
        compute_metrics=compute_metrics
    )

    print(f"Fold {fold_idx} training begins…\n")
    trainer.train()

    print(f"\n--- Performance evaluation for Fold {fold_idx} ---")
    preds_output = trainer.predict(tokenized_eval)
    logits = preds_output.predictions
    y_pred = np.argmax(logits, axis=1)
    y_true = preds_output.label_ids

    rpt = classification_report(y_true, y_pred, target_names=["General", "Critical"], output_dict=True)
    all_reports.append(rpt)
    f1_critical_scores.append(rpt["Critical"]["f1-score"])

    print(classification_report(y_true, y_pred, target_names=["General", "Critical"]))

# Cross-Validation Summary
print("\n\n===== Summary Cross-Validation =====")
for i, rpt in enumerate(all_reports, 1):
    f1c = rpt["Critical"]["f1-score"]
    print(f"  Fold {i}: F1 (Critical) = {f1c:.3f}")

avg_f1_crit = np.mean(f1_critical_scores)
print(f"\nAverage F1 (Critical) over 5 folds = {avg_f1_crit:.3f}")

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bionlp/bluebert_pubmed_mimic_uncased_L-12_H-768_A-12 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-15-816668dbdcfb>:187: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalTrainer.__init__`. Use `processing_class` instead.
  trainer = FocalTrainer(



Fold 5 training begins...


Step,Training Loss
50,0.039900
100,0.036500
150,0.027100
200,0.019700
250,0.008500
300,0.003800
350,0.001700
400,0.000900
450,0.000400



--- Performance evaluation for Fold 5 ---


              precision    recall  f1-score   support

     General       0.94      0.94      0.94        86
    Critical       0.86      0.86      0.86        36

    accuracy                           0.92       122
   macro avg       0.90      0.90      0.90       122
weighted avg       0.92      0.92      0.92       122



 Summary Cross-Validation:
  Fold 1: F1 (Critical) = 0.725
  Fold 2: F1 (Critical) = 0.744
  Fold 3: F1 (Critical) = 0.848
  Fold 4: F1 (Critical) = 0.727
  Fold 5: F1 (Critical) = 0.861

 Average F1 (Critical) over 5 folds = 0.781


In [16]:
print("\n===== Classification Report (Test Set) =====")
print(classification_report(y_true, y_pred, target_names=["General", "Critical"]))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=[0,1]
)
acc = accuracy_score(y_true, y_pred)
print(f"\nAccuracy (Test)         = {acc:.4f}")
print(f"F1 (General, label=0)   = {f1[0]:.4f}")
print(f"F1 (Critical, label=1)  = {f1[1]:.4f}")


===== Classification Report (Test Set) =====
              precision    recall  f1-score   support

     General       0.94      0.94      0.94        86
    Critical       0.86      0.86      0.86        36

    accuracy                           0.92       122
   macro avg       0.90      0.90      0.90       122
weighted avg       0.92      0.92      0.92       122


Accuracy (Test)         = 0.9180
F1 (General, label=0)   = 0.9419
F1 (Critical, label=1)  = 0.8611


In [17]:
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted"
)
acc = accuracy_score(y_true, y_pred)

print("\n=== Overall (weighted) Metrics ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {precision_w:.4f}")
print(f"Recall:    {recall_w:.4f}")
print(f"F1-score:  {f1_w:.4f}")


=== Overall (weighted) Metrics ===
Accuracy:  0.9180
Precision: 0.9180
Recall:    0.9180
F1-score:  0.9180
